In [1]:
import pandas as pd
from tqdm import tqdm

In [2]:
tqdm.pandas()

In [3]:
## yelp_onnx_val_results 

yelp_onnx_val_results = pd.read_json("/Users/cgopal/Downloads/YELP_NER_ONNX_VAL_RESULTS/ML_output_YELP_KEYWORDS_DATA.json")
print(len(yelp_onnx_val_results))

2870


In [4]:
yelp_onnx_val_results

,query,intent,city,state
0,24 hour cleaning services,yelp_intent,None,None
1,24 hour maid service,yelp_intent,None,None
2,24 hour pharmacy,yelp_intent,None,None
3,24 hour restaurants,yelp_intent,None,None
4,24 hour walmart,navigation_intent,##mart,None
...,...,...,...,...
2865,yoga,yelp_intent,None,None
2866,yogurstory,yelp_intent,None,None
2867,you move me,yelp_intent,None,None
2868,yume wo katare,yelp_intent,yu katare,None


In [5]:
yelp_onnx_val_results['intent'].value_counts()['yelp_intent']/len(yelp_onnx_val_results)

0.926829268292683

In [6]:
## yelp_onnx_native_val_results 

yelp_onnx_native_val_results = pd.read_json("/Users/cgopal/Downloads/YELP_NER_ONNX_NATIVE_VAL_RESULTS/ML_output_YELP_KEYWORDS_DATA.json")
print(len(yelp_onnx_native_val_results))

2870


In [7]:
yelp_onnx_native_val_results

,query,intent,city,state
0,24 hour cleaning services,yelp_intent,None,None
1,24 hour maid service,yelp_intent,None,None
2,24 hour pharmacy,yelp_intent,None,None
3,24 hour restaurants,yelp_intent,None,None
4,24 hour walmart,navigation_intent,##mart,None
...,...,...,...,...
2865,yoga,yelp_intent,None,None
2866,yogurstory,yelp_intent,None,None
2867,you move me,yelp_intent,None,None
2868,yume wo katare,yelp_intent,yu katare,None


In [8]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import numpy as np

# Load tokenizer and model
intent_tokenizer = AutoTokenizer.from_pretrained("Mozilla/mobilebert-uncased-finetuned-LoRA-intent-classifier")
intent_model = AutoModelForSequenceClassification.from_pretrained("Mozilla/mobilebert-uncased-finetuned-LoRA-intent-classifier")

# ID to label map
id2label = {
    0: 'information_intent',
    1: 'yelp_intent',
    2: 'navigation_intent',
    3: 'travel_intent',
    4: 'purchase_intent',
    5: 'weather_intent',
    6: 'translation_intent',
    7: 'unknown'
}

# Prepare input (note: return_tensors="pt" for PyTorch)
def predict(query):
    inputs = intent_tokenizer(
        query,
        return_tensors="pt",
        padding="max_length",
        truncation=True,
        max_length=64
    )

    # Forward pass through model
    with torch.no_grad():
        outputs = intent_model(**inputs)

    logits = outputs.logits
    prediction = torch.argmax(logits, dim=1).item()
    probabilities = torch.nn.functional.softmax(logits, dim=1).numpy()
    rounded_probabilities = np.round(probabilities, decimals=3)

    # Output result
    pred_result = id2label[prediction]
    return pred_result
#     print("Prediction:", pred_result)
#     print("Probabilities:", rounded_probabilities)


In [9]:
# predict("24 hour cleaning services")
python_preds = yelp_onnx_native_val_results['query'].progress_apply(predict)

100%|██████████████████| 2870/2870 [00:54<00:00, 52.56it/s]


In [10]:
python_preds.value_counts(normalize=True)['yelp_intent']

0.9383275261324042

In [11]:
yelp_onnx_native_val_results['intent'].value_counts()['yelp_intent']/len(yelp_onnx_native_val_results)

0.9264808362369338

In [12]:
print("Missed when we move to onnx_native")
yelp_onnx_val_results.loc[
    (yelp_onnx_val_results['intent'] != yelp_onnx_native_val_results['intent'])
    & (yelp_onnx_val_results['intent'] == 'yelp_intent')
]

Missed when we move to onnx_native


,query,intent,city,state
515,crepes,yelp_intent,None,None
1668,mokkoji,yelp_intent,mo,None
2213,professional pool table movers,yelp_intent,None,None
2619,son and garden,yelp_intent,None,None


In [13]:
print("onnx_native got it right but onnx misclassified")
yelp_onnx_native_val_results.loc[
    (yelp_onnx_val_results['intent'] != yelp_onnx_native_val_results['intent'])
    & (yelp_onnx_native_val_results['intent'] == 'yelp_intent')
]

onnx_native got it right but onnx misclassified


,query,intent,city,state
1475,keens steakhouse,yelp_intent,None,None
1794,moving price,yelp_intent,None,None
2119,pottery classes,yelp_intent,None,None


#### NER onnx vs NER onnx_native comparisons

In [14]:
ner_onnx_val_results = pd.read_json("/Users/cgopal/Downloads/YELP_NER_ONNX_VAL_RESULTS/ML_output_NER_VAL_DATA.json")
print(len(ner_onnx_val_results))
ner_onnx_val_results

1000


,query,intent,city,state
0,train stations near detroit,yelp_intent,detroit,None
1,voter registration in wa,navigation_intent,None,wa
2,"weather report for los angeles, california",weather_intent,los angeles,california
3,city hall in boston,yelp_intent,boston,None
4,"bike rentals in baltimore, maryland",yelp_intent,baltimore,maryland
...,...,...,...,...
995,tax offices in las vegas,yelp_intent,las vegas,None
996,"homes for rent in baltimore, maryland",yelp_intent,baltimore,maryland
997,"top universities in san diego, california",yelp_intent,san diego,california
998,"top universities in tampa, florida",yelp_intent,tampa,florida


In [15]:
ner_onnx_native_val_results = pd.read_json("/Users/cgopal/Downloads/YELP_NER_ONNX_NATIVE_VAL_RESULTS/ML_output_NER_VAL_DATA.json")
print(len(ner_onnx_native_val_results))
ner_onnx_native_val_results

1000


,query,intent,city,state
0,train stations near detroit,yelp_intent,detroit,None
1,voter registration in wa,navigation_intent,None,wa
2,"weather report for los angeles, california",weather_intent,los angeles,california
3,city hall in boston,yelp_intent,boston,None
4,"bike rentals in baltimore, maryland",yelp_intent,baltimore,maryland
...,...,...,...,...
995,tax offices in las vegas,yelp_intent,las vegas,None
996,"homes for rent in baltimore, maryland",yelp_intent,baltimore,maryland
997,"top universities in san diego, california",yelp_intent,san diego,california
998,"top universities in tampa, florida",yelp_intent,tampa,florida


In [16]:
ner_onnx_val_results.loc[ner_onnx_val_results['city'].fillna('') != ner_onnx_native_val_results['city'].fillna('')]

,query,intent,city,state
835,philadelphia airport information,navigation_intent,philadelphia,None


In [17]:
ner_onnx_native_val_results.loc[ner_onnx_val_results['city'].fillna('') != ner_onnx_native_val_results['city'].fillna('')]

,query,intent,city,state
835,philadelphia airport information,navigation_intent,philadelphia information,None


In [18]:
ner_onnx_val_results.loc[ner_onnx_val_results['state'].fillna('') != ner_onnx_native_val_results['state'].fillna('')]


,query,intent,city,state


In [19]:
ner_onnx_native_val_results.loc[ner_onnx_val_results['state'].fillna('') != ner_onnx_native_val_results['state'].fillna('')]


,query,intent,city,state


In [20]:
ner_onnx_val_results.loc[
    (ner_onnx_val_results['city'].fillna('') != ner_onnx_native_val_results['city'].fillna(''))
    | (ner_onnx_val_results['state'].fillna('') != ner_onnx_native_val_results['state'].fillna(''))
]


,query,intent,city,state
835,philadelphia airport information,navigation_intent,philadelphia,None


In [24]:
(
    (ner_onnx_val_results['city'].fillna('') == ner_onnx_native_val_results['city'].fillna(''))
    & (ner_onnx_val_results['state'].fillna('') == ner_onnx_native_val_results['state'].fillna(''))
).sum()/len(ner_onnx_val_results)


0.999

In [25]:
ner_onnx_val_results

,query,intent,city,state
0,train stations near detroit,yelp_intent,detroit,None
1,voter registration in wa,navigation_intent,None,wa
2,"weather report for los angeles, california",weather_intent,los angeles,california
3,city hall in boston,yelp_intent,boston,None
4,"bike rentals in baltimore, maryland",yelp_intent,baltimore,maryland
...,...,...,...,...
995,tax offices in las vegas,yelp_intent,las vegas,None
996,"homes for rent in baltimore, maryland",yelp_intent,baltimore,maryland
997,"top universities in san diego, california",yelp_intent,san diego,california
998,"top universities in tampa, florida",yelp_intent,tampa,florida
